# Colab GPU Trong IDE (Git Sync): Train LoRA Minh Khang + Giọng Biểu Cảm + Video Hành Động Wan 2.2

Đã tối ưu chạy mượt trên cả **Tesla T4 (15GB)** lẫn **NVIDIA L4 (24GB)**, loại bỏ lỗi build `scipy`/`oyaml` của `ai-toolkit`:
1. **Cell 1:** Pull code & 12 ảnh mới nhất từ GitHub (`nguyenkhanhan123/Test`) + kiểm tra GPU.
2. **Cell 2:** Cài `diffusers + peft + bitsandbytes` đồng bộ và train SDXL LoRA (`pytorch_lora_weights.safetensors`, ~5–8 phút).
3. **Cell 3:** Dùng LoRA vừa train sinh Keyframe hành động mới cho `shot_103`, `shot_104`.
4. **Cell 4:** Audio-First — tạo giọng biểu cảm & khóa thời lượng từng shot.
5. **Cell 5:** Render cảnh hành động thật bằng `Wan-AI/Wan2.2-TI2V-5B-Diffusers` (tự chỉnh FP16 + sequential offload trên T4).
6. **Cell 6:** Ghép video hoàn chỉnh, phát xem ngay trong IDE & tự động ngắt GPU.

In [4]:
# CELL 1: ĐỒNG BỘ GIT + KIỂM TRA GPU + CÀI THƯ VIỆN CỐT LÕI
import os
REPO_URL = 'https://github.com/nguyenkhanhan123/Test.git'
REPO_DIR = '/content/Test'
PROJECT_DIR = '/content/Test/ai_film_pipeline'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin main && git reset --hard origin/main

%cd {PROJECT_DIR}
!nvidia-smi
!pip uninstall -y -q torchao
!pip install -q --upgrade edge-tts pysubs2 pydantic imageio-ffmpeg mutagen peft bitsandbytes accelerate transformers git+https://github.com/huggingface/diffusers.git
!python prepare_user_assets.py --trigger nv_minhkhang

/content/Test
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 10 (delta 6), reused 10 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 5.86 KiB | 1.95 MiB/s, done.
From https://github.com/nguyenkhanhan123/Test
 * branch            main       -> FETCH_HEAD
   4febc5d..be4307b  main       -> origin/main
HEAD is now at be4307b Fix Colab T4 LoRA training (diffusers+peft), Wan 2.2 T4 memory, and Linux subtitles
/content/Test/ai_film_pipeline
Sat Nov 28 12:19:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |

In [5]:
# CELL 2: TRAIN CHARACTER LORA TỪ 12 ẢNH (DIFFUSERS + PEFT + ADAMW8BIT — TƯƠNG THÍCH T4 & L4)
%cd /content/Test/ai_film_pipeline
!python cloud_scripts/train_sdxl_lora_peft.py \
  --dataset user_inputs/photos \
  --output loras/nv_minhkhang_lora_v1 \
  --steps 360 \
  --resolution 768

/content/Test/ai_film_pipeline
Disabling torchao because diffusers requires `torchao>=0.15.0` but found version 0.10.0. Upgrade it with `pip install -U 'torchao>=0.15.0'`.
[LoRA] Dataset: 12 images | res=768 | steps=360 | dtype=torch.bfloat16
tokenizer_config.json: 100% 737/737 [00:00<00:00, 3.30MB/s]
vocab.json: 100% 1.06M/1.06M [00:00<00:00, 55.9MB/s]
merges.txt: 100% 525k/525k [00:00<00:00, 113MB/s]
special_tokens_map.json: 100% 472/472 [00:00<00:00, 3.17MB/s]
tokenizer_config.json: 100% 725/725 [00:00<00:00, 3.67MB/s]
special_tokens_map.json: 100% 460/460 [00:00<00:00, 2.79MB/s]
config.json: 100% 565/565 [00:00<00:00, 3.58MB/s]

text_encoder/model.safetensors: downloading bytes:  34% 166M/492M [00:01<00:01, 233MB/s, 12.6MB/s  ]  
text_encoder/model.safetensors: reconstructing file:   0% 269k/492M [00:01<40:56, 200kB/s]
text_encoder/model.safetensors: downloading bytes:  63% 310M/492M [00:01<00:00, 278MB/s, 25.6MB/s  ]s  ]
text_encoder/model.safetensors: downloading bytes: 100% 326M

In [6]:
# CELL 3: DÙNG LORA VỪA TRAIN SINH KEYFRAME HÀNH ĐỘNG MỚI CHO SHOT_103 & SHOT_104
%cd /content/Test/ai_film_pipeline
!pip uninstall -y -q torchao
!python cloud_scripts/step4_gen_keyframes_lora.py

/content/Test/ai_film_pipeline


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading default_0 was unsuccessful with the following error: 
Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported


ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [ ]:
# CELL 4: AUDIO-FIRST — TẠO GIỌNG BIỂU CẢM & KHÓA THỜI LƯỢNG TỪNG SHOT
%cd /content/Test/ai_film_pipeline
from pathlib import Path
voices_ready = list(Path('user_inputs/voices').glob('*.wav'))
if voices_ready:
    !pip install -q git+https://github.com/danhtran2mind/F5-TTS.git whisperx
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='danhtran2mind/Vi-F5-TTS', local_dir='/content/Vi-F5-TTS')
    from cloud_scripts.step3_cloud_vi_f5_tts import clone_episode_voices_vi_f5
    clone_episode_voices_vi_f5(Path('.'), Path('script_action_demo.json'), Path('/content/Vi-F5-TTS'))
else:
    print('Dùng Edge TTS biểu cảm đa sắc thái (thì thầm / căng thẳng / hô chiến đấu / lắng dịu)...')
    !python step_action_demo.py --tts-only

In [ ]:
# CELL 5: RENDER VIDEO HÀNH ĐỘNG THẬT BẰNG WAN 2.2 TI2V-5B
%cd /content/Test/ai_film_pipeline
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python cloud_scripts/step5_batch_wan22_diffusers.py --script script_action_demo.json --steps 24

In [ ]:
# CELL 6: GHÉP VIDEO HOÀN CHỈNH, PHÁT XEM NGAY TRONG IDE & TỰ ĐỘNG NGẮT GPU
%cd /content/Test/ai_film_pipeline
from IPython.display import Video, display

try:
    !python step_action_demo.py --skip-tts
    display(Video('output/action_pilot_demo.mp4', embed=True, width=960))
finally:
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass